In [4]:
import cv2
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATA_PATH = ## folder location for patches and masks

class TiDataset(Dataset):
    def __init__(self, root_dir): 
        self.images_dir = os.path.join(root_dir, "patches")
        self.masks_dir = os.path.join(root_dir, "masks")
        self.ids = [f.replace("_mask.png", "") for f in os.listdir(self.masks_dir) if f.endswith("_mask.png")]
        
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, i):
        # Now the base ID is clean, these will construct correctly
        img_name = self.ids[i] + ".jpg"
        mask_name = self.ids[i] + "_mask.png"

        # 1. Load Image
        image_path = os.path.join(self.images_dir, img_name)
        image = cv2.imread(image_path)
        
        # Fallback if jpg doesn't exist
        if image is None:
             img_name = self.ids[i] + ".png"
             image_path = os.path.join(self.images_dir, img_name)
             image = cv2.imread(image_path)
             
        # FIX: Safety check in case the fallback also fails
        if image is None:
            raise RuntimeError(f"Could not read image file for ID: {self.ids[i]} at {image_path}")
        
        # Convert to RGB
        if image.ndim == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. Load Mask
        mask_path = os.path.join(self.masks_dir, mask_name)
        mask = cv2.imread(mask_path, 0) # 0 = cv2.IMREAD_GRAYSCALE
        
        if mask is None:
            raise RuntimeError(f"Could not read mask file: {mask_path}")
        
        # --- RESIZE ---
        image = cv2.resize(image, (256, 256))
        # Use INTER_NEAREST for masks to avoid creating new decimal values (keep it 0 or 255)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

        # 3. Format for PyTorch
        # Image: (H, W, C) -> (C, H, W), Normalized 0-1
        image = image.transpose(2, 0, 1).astype('float32') / 255.0
        
        # Mask: Binary 0 or 1, Shape (1, H, W)
        mask = (mask > 128).astype('float32')
        mask = np.expand_dims(mask, 0)
        
        #Convert numpy arrays to PyTorch tensors explicitly
        image_tensor = torch.from_numpy(image)
        mask_tensor = torch.from_numpy(mask)
        
        return image_tensor, mask_tensor

# Re-initialize the dataset
if __name__ == "__main__":


    
    if not os.path.exists(DATA_PATH):
        print(f"Warning: The path {DATA_PATH} does not exist.")
    else:
        dataset = TiDataset(DATA_PATH) 
        print(f"Dataset ready. Training on {len(dataset)} annotated images.")
        

Dataset ready. Training on 2 annotated images.


In [5]:
import segmentation_models_pytorch as smp
import torch

# Create Model
model = smp.Unet(
    encoder_name="resnet18",        
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=1,                      
    activation=None  # <--- Set to None. We let the Loss Function handle the Sigmoid.
)

# Training Settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# DiceLoss with from_logits=True is numerically more stable
loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)

dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Training Loop
epochs = 50
print(f"Starting Training on {device}...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for images, masks in dataloader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (Outputs are raw logits now)
        logits = model(images)
        
        # Calculate loss
        loss = loss_fn(logits, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")

print("✅ Training Complete.")
torch.save(model.state_dict(), "titanium_unet_weights.pth")

Starting Training on cpu...
Epoch 1/50, Loss: 0.3929
Epoch 2/50, Loss: 0.3789
Epoch 3/50, Loss: 0.3616
Epoch 4/50, Loss: 0.3478
Epoch 5/50, Loss: 0.3348
Epoch 6/50, Loss: 0.3235
Epoch 7/50, Loss: 0.3134
Epoch 8/50, Loss: 0.3044
Epoch 9/50, Loss: 0.2966
Epoch 10/50, Loss: 0.2895
Epoch 11/50, Loss: 0.2832
Epoch 12/50, Loss: 0.2776
Epoch 13/50, Loss: 0.2722
Epoch 14/50, Loss: 0.2673
Epoch 15/50, Loss: 0.2627
Epoch 16/50, Loss: 0.2585
Epoch 17/50, Loss: 0.2544
Epoch 18/50, Loss: 0.2506
Epoch 19/50, Loss: 0.2470
Epoch 20/50, Loss: 0.2436
Epoch 21/50, Loss: 0.2404
Epoch 22/50, Loss: 0.2374
Epoch 23/50, Loss: 0.2344
Epoch 24/50, Loss: 0.2316
Epoch 25/50, Loss: 0.2289
Epoch 26/50, Loss: 0.2263
Epoch 27/50, Loss: 0.2238
Epoch 28/50, Loss: 0.2214
Epoch 29/50, Loss: 0.2192
Epoch 30/50, Loss: 0.2170
Epoch 31/50, Loss: 0.2148
Epoch 32/50, Loss: 0.2128
Epoch 33/50, Loss: 0.2108
Epoch 34/50, Loss: 0.2089
Epoch 35/50, Loss: 0.2070
Epoch 36/50, Loss: 0.2052
Epoch 37/50, Loss: 0.2035
Epoch 38/50, Loss: 

In [ ]:
# Pick a random patch to test (even one you didn't annotate)
import matplotlib.pyplot as plt
test_img_path = f"{DATA_PATH}/patches/patch_1_3.jpg" # Change to any patch
image = cv2.imread(test_img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Preprocess
x = image_rgb.transpose(2, 0, 1).astype('float32') / 255.0
x = torch.from_numpy(x).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    pred_mask = model(x)
    pred_mask = (pred_mask > 0.5).float().cpu().numpy()[0][0]

# Display
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original SEM")
plt.subplot(1, 2, 2)
plt.imshow(pred_mask, cmap='gray')
plt.title("U-net Prediction (Alpha Phase)")
plt.show()